# Seed43: SCST + KL500, matched VALID5000 only

Fresh from own best XE179, frozen own-XE teacher, beta1.0,500updates/global8/RNG4342, no CE/object reward.
Compares XE, saved pure-SCST500 from185 and new KL500 with the same cached VALID, beam5 and frozen0.3/0.4/0.3 rerank.
Reports all8metrics and NLL; no TEST, coefficient sweep or automatic extension.

Run on the ducanh3 account with T4 x2, Internet and Kaggle Secret codex or KAGGLE_API_TOKEN.
Attach only csa-scst-full-karpathy-cache-v1, mscoco-2014 and objectdetectionecache.
Exact XE/control weights download by API. Cached patch arrays are linked readonly.
The tensor/CUDA preflight and first-step teacher/on-policy guards must pass before accepting the run.
Keep the entire output. Send csa_seed43_scst_kl500_valid5000_review.zip. Three-hour STOP cap, not an ETA.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='aae9afc7966ee4aa36dd67c4b4839ff788e775fe'
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
PROMPT_TEST_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-test-cache-v181')
CHECKPOINT_ROOT='' # API-only exact best.bin; no checkpoint mount needed.
WALL_HOURS=3 # STOP cap including CPU metrics, not ETA or billing.
OUTPUT=Path('/kaggle/working/csa_seed43_scst_kl500_valid5000')
for p in (COCO_JSON,):
    if not p.exists():raise FileNotFoundError('Attach/correct original inputs: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve existing captions/output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<8*1024**3:raise RuntimeError('Need8GiB free for TEST patches/checkpoint/CPU scorer assets')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch199-test-',dir='/tmp'));DEPS=SOURCE/'deps';CLI=SOURCE/'cli';CHAIR=SOURCE/'chair';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(semantic=True,prompt_test_root=str(PROMPT_TEST_ROOT),output=str(OUTPUT),stage=str(SOURCE/'test_patches'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),checkpoint_root=CHECKPOINT_ROOT,chair=str(CHAIR),worker_seconds=1800)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline before download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned input: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

TRAIN_REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
TRAIN_PINS={'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09'}

SEED=43
SOURCE_VERSION='ducanh3/notebook7bc83c4c76/179'
SECRET_NAMES=('codex','KAGGLE_API_TOKEN')

DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
if not DETECTIONS.is_file():raise FileNotFoundError('Attach objectdetectionecache or correct DETECTIONS')
os.environ.update(TOKENIZERS_PARALLELISM='false',USE_TF='0',USE_FLAX='0',HF_HUB_DISABLE_XET='1')

ROOT_OUTPUT=OUTPUT
CACHE_ROOT='/kaggle/input/datasets/ducanh3/csa-scst-full-karpathy-cache-v1'
if not Path(CACHE_ROOT).is_dir():raise FileNotFoundError('Attach full SCST cache or correct CACHE_ROOT')
cfg.update(train_stage=str(SOURCE/'pilot_train'),train_seconds=3600)
write_config()

CONTROL_VERSION='ducanh3/notebook7bc83c4c76/185'
KL_BETA=1.0


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('TEST parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('TEST/CPU metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('TEST parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
ROUTING_PREFLIGHT = "import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output(SOURCE_VERSION,path=directory)\n    request=endpoint.call_args.args[0]\n    owner,slug,version=SOURCE_VERSION.split('/')\n    assert (request.user_name,request.kernel_slug,request.version_label)==(owner,slug,'v'+version)\nprint('Exact TEST source API routing PASS',flush=True)\n"
complete = False
try:
    for path, name in MAPPING.items():
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/' + REVISION + '/' + path, HASHES[path], SOURCE / name)
    for path, digest in TRAIN_PINS.items():
        name = 'bank_layers.py' if path == 'experiments/csa_paper/model.py' else Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/' + TRAIN_REVISION + '/' + path, digest, SOURCE / name)
    (SOURCE / 'csa_per_caption_test_data.py').write_text('"""Fixed best9 TEST prompt/patch contracts; NumPy only, no model execution."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read, save, sha\nimport test199_data as shared\nfrom semantic_data import ARRAYS, record_arrays, checked_root\nHERE = Path(__file__).resolve().parent\nCHECKPOINT_SHA = \'a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5\'\nVERSION = \'ducanh3/notebook7bc83c4c76/179\'\n\ndef validate_lock(value):\n    baseline = valid_cohort()\n    if any((value[k] != baseline[k] for k in (\'expected_test_ids\', \'test_id_sha256\', \'beam_width\', \'max_new_tokens\', \'length_normalization\', \'reranking\'))):\n        raise ValueError(\'Changed fixed TEST cohort/decode\')\n    if value[\'source_version\'] != VERSION or value[\'checkpoint_sha256\'] != CHECKPOINT_SHA or value[\'epoch\'] != 8 or (value[\'updates\'] != 141688) or (value[\'method\'] != \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\'):\n        raise ValueError(\'Need fixed seed43 best epoch8\')\n    return value\n\ndef valid_cohort():\n    import hashlib\n    value = shared.lock()\n    value.update(expected_test_ids=VALID_IDS, test_id_sha256=hashlib.sha256(json.dumps(VALID_IDS, separators=(\',\', \':\')).encode()).hexdigest())\n    return value\n\ndef lock():\n    value = valid_cohort()\n    value.update(schema=\'csa_per_caption_seed_test_v1\', source_version=VERSION, checkpoint_sha256=CHECKPOINT_SHA, training_commit=\'5eb012279e6fd0b5e47a56710ed5e3efa7449d2f\', epoch=8, updates=141688, method=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\')\n    return validate_lock(value)\n\ndef checkpoint(root):\n    root = Path(root)\n    files = [root] if root.is_file() else sorted(root.rglob(\'best.bin\'))\n    matches = [p for p in files if p.is_file() and sha(p) == CHECKPOINT_SHA]\n    if not matches:\n        raise ValueError(\'Missing exact seed43 best.bin SHAa51e7ec2\')\n    return matches[0]\n\ndef runtime_policy(training, current):\n    allowed = (\'2.11.0+cu128\', \'2.11.0+cu130\')\n    if training not in allowed or current not in allowed:\n        raise RuntimeError(\'Unreviewed Torch build; keep log, do not blindly reinstall.\')\n    return dict(training_torch=training, inference_torch=current, cuda_wheel_changed=training != current, policy=\'Recorded training and inference Torch builds; FP32 and strict checkpoint checks.\', bitwise_cross_build_equivalence_established=False)\n\ndef validate_checkpoint_state(ck):\n    s, i = (ck[\'state\'], ck[\'identity\'])\n    expected = lock()\n    if s[\'epoch\'] != 9 or s[\'batch_cursor\'] != 0 or s[\'phase\'] != \'train\' or (s[\'updates\'] != 141688) or (s[\'caption_events\'] != 4533976) or (s[\'best_epoch\'] != 8) or (len(s[\'history\']) != 8):\n        raise ValueError(\'Wrong seed43 best snapshot cursor\')\n    if i[\'source\'][\'commit\'] != expected[\'training_commit\'] or i[\'method\'] != expected[\'method\'] or i[\'epochs\'] != 10 or (i[\'global_batch\'] != 32):\n        raise ValueError(\'Wrong checkpoint architecture/training identity\')\n    if i.get(\'seed\') != 43:\n        raise ValueError(\'Wrong training seed\')\n    if not {\'core.nouns\', \'core.verbs\'} <= set(ck[\'model\']):\n        raise ValueError(\'Missing frozen TRAIN banks in checkpoint buffers\')\n    return expected\n\ndef configure_valid(cfg):\n    global VALID_IDS\n    if sha(cfg[\'dataset_json\']) != shared.DATA_SHA:\n        raise ValueError(\'Changed Karpathy metadata\')\n\n    def val_rows(ds, locked):\n        rows = sorted((r for r in ds[\'images\'] if r[\'split\'] == \'val\'), key=lambda r: int(r[\'cocoid\']))\n        if [int(r[\'cocoid\']) for r in rows] != locked[\'expected_test_ids\']:\n            raise ValueError(\'Wrong VALID IDs\')\n        return rows\n    VALID_IDS = sorted((int(r[\'cocoid\']) for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\'] == \'val\'))\n    if len(VALID_IDS) != 5000 or len(set(VALID_IDS)) != 5000:\n        raise ValueError(\'Need VALID5000\')\n    shared.selected_test_rows = val_rows\n\ndef prompt_inputs(cfg):\n    configure_valid(cfg)\n    inv = read(HERE / \'FULL_PROMPT_FRESH_V175_RESULT.json\')[\'transport_inventory\'][\'cache_files\']\n    wanted = {n: inv[n] for n in (\'cache_receipt.json\', \'cohort.json\', \'prompt_links.jsonl\', \'prompt_texts.json\', \'prompt_embeddings.npy\')}\n    root = checked_root(cfg[\'prompt_test_root\'], \'cache_receipt.json\', wanted)\n    cohort = read(root / \'cohort.json\')\n    if cohort[\'test\'] or sorted(cohort[\'val\']) != VALID_IDS:\n        raise ValueError(\'Wrong prompt175 VALID cohort\')\n    texts = read(root / \'prompt_texts.json\')\n    vectors = np.load(root / \'prompt_embeddings.npy\', allow_pickle=False)\n    if texts != sorted(set(texts)) or vectors.shape != (410, 512) or vectors.dtype != np.float32 or (not np.isfinite(vectors).all()):\n        raise ValueError(\'Changed prompt175 embeddings\')\n    with (root / \'prompt_links.jsonl\').open(encoding=\'utf8\') as stream:\n        rows = [r for r in map(json.loads, stream) if r[\'group\'] == \'val\']\n    if sorted((r[\'coco_id\'] for r in rows)) != VALID_IDS:\n        raise ValueError(\'Wrong VALID prompt links\')\n    return (rows, texts, vectors, wanted)\n\ndef prepare(cfg):\n    rows, texts, vectors, seals = prompt_inputs(cfg)\n    for row in rows:\n        record_arrays(row, texts)\n    shared.prepare(cfg, locked=lock(), weight_path=checkpoint(cfg[\'checkpoint_root\']))\n    stage_prompts(cfg, rows, texts, vectors, seals)\n\ndef stage_prompts(cfg, rows, texts, vectors, seals):\n    encoded = [record_arrays(row, texts) for row in rows]\n    stage = Path(cfg[\'stage\'])\n    out = Path(cfg[\'output\'])\n    ids = np.load(stage / \'ids.npy\', allow_pickle=False)\n    lookup = {r[\'coco_id\']: j for j, r in enumerate(rows)}\n    order = [lookup[int(i)] for i in ids]\n    for key in ARRAYS:\n        np.save(stage / (\'sem_\' + key + \'.npy\'), np.stack([encoded[j][key] for j in order]))\n    np.save(stage / \'sem_text_vectors.npy\', vectors)\n    receipt = read(out / \'stage_receipt.json\')\n    receipt[\'method\'] = \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\'\n    receipt[\'prompt175_files_sha256\'] = seals\n    receipt[\'stage_files_sha256\'].update({p.name: sha(p) for p in stage.glob(\'sem_*.npy\')})\n    save(out / \'stage_receipt.json\', receipt)\n    saved = read(out / \'test_lock.json\')\n    saved[\'stage_receipt_sha256\'] = sha(out / \'stage_receipt.json\')\n    save(out / \'test_lock.json\', saved)\n    save(out / \'semantic_test_stage.json\', dict(images=5000, objects=sum((len(r[\'objects\']) for r in rows)), relations=sum((len(r[\'relations\']) for r in rows)), text_vectors=410, geometry=\'same training record_arrays ROI box->fractional14x14patch support; endpoint IDs\', TRAIN_TEST_features_read=False, old_ROI_features_used=False, old_vocabulary_used=False))\n\ndef load(cfg, verify=True):\n    result = shared.load(cfg, verify=verify, validator=validate_lock)\n    if result[2].get(\'method\') != \'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\':\n        raise ValueError(\'Semantic TEST stage incomplete\')\n    return result\n\ndef bind_verified_cache(cfg, cache_source_sha):\n    configure_valid(cfg)\n    out = Path(cfg[\'output\'])\n    cached = read(out/\'test_lock.json\')\n    expected = lock()\n    same = (\'expected_test_ids\', \'test_id_sha256\', \'beam_width\', \'max_new_tokens\',\n            \'length_normalization\', \'reranking\', \'training_commit\', \'method\')\n    if any(cached[k] != expected[k] for k in same):\n        raise ValueError(\'Cache cohort, decoder or architecture differs\')\n    if cached[\'checkpoint_sha256\'] != cache_source_sha:\n        raise ValueError(\'Wrong original cache source\')\n    rebound = dict(cached, **expected)\n    rebound[\'checkpoint\'] = cfg[\'checkpoint_root\']\n    validate_lock(rebound)\n    save(out/\'cache_checkpoint_binding.json\', dict(\n        original_cache_source_sha256=cache_source_sha, xe_checkpoint_sha256=CHECKPOINT_SHA,\n        checkpoint_source=VERSION, original_lock=cached, bound_lock=rebound,\n        cache_payloads_changed=False, training_references_scope=\'TRAIN/restval only\'))\n    save(out/\'test_lock.json\', rebound)\n', encoding='utf8')
    (SOURCE / 'run_csa_per_caption_test.py').write_text('"""USER-RUN fixed199 inference and CPU scoring. No training mode."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom test199_data import prepare,load,lock,validate_lock,checkpoint,runtime_policy,CHECKPOINT_SHA,TEST_SIZE\nfrom saved_quality_data import check_rows,chair_counts\n\ndef contracts(cfg):\n    if cfg.get(\'semantic\'):\n        import csa_per_caption_test_data as module\n        module.configure_valid(cfg)\n    else:\n        import test199_data as module\n    return module\n\n\ndef checked_weights(cfg):\n    import torch\n    d=contracts(cfg)\n    if cfg.get(\'evaluate_pilot\'):\n        receipt=read(Path(cfg[\'output\'])/\'pilot_checkpoint_receipt.json\')\n        path=Path(cfg[\'eval_checkpoint\'])\n        if receipt[\'updates\']!=cfg[\'eval_update\'] or receipt[\'source_sha256\']!=d.CHECKPOINT_SHA or sha(path)!=receipt[\'sha256\']:\n            raise ValueError(\'Wrong SCST-only checkpoint\')\n        ck=torch.load(path,map_location=\'cpu\',weights_only=False)\n        if ck[\'updates\']!=cfg[\'eval_update\'] or ck[\'source_sha256\']!=d.CHECKPOINT_SHA:\n            raise ValueError(\'Pilot state differs\')\n        return ck,d.runtime_policy(read(Path(cfg[\'output\'])/\'training_runtime.json\')[\'torch_version\'],str(torch.__version__))\n    d=contracts(cfg);ck=torch.load(d.checkpoint(cfg[\'checkpoint_root\']),map_location=\'cpu\',weights_only=False)\n    if cfg.get(\'semantic\'):\n        d.validate_checkpoint_state(ck)\n        return ck,d.runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n    if ck[\'state\'][\'updates\']!=177110 or ck[\'state\'][\'best_epoch\']!=10 or ck[\'identity\'][\'source\'][\'commit\']!=lock()[\'training_commit\']:raise ValueError(\'Unexpected fixed199 training state\')\n    return ck,runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n\ndef checkpoint_preflight(cfg):\n    ck,policy=checked_weights(cfg);del ck\n    save(Path(cfg[\'output\'])/\'checkpoint_runtime_preflight.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,runtime_policy=policy,models_executed=False))\n    print(\'Checkpoint/runtime metadata PASS:\',policy,flush=True)\n\ndef worker(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,check_paths,text_from_ids\n    from patch_data import batch as caption_batch\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Use T4x2\')\n    torch.set_num_threads(1);torch.cuda.set_device(rank);device=torch.device(\'cuda\',rank)\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    dist.init_process_group(\'nccl\',timeout=timedelta(seconds=1200));started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        d=contracts(cfg);data,words,receipt,locked=d.load(cfg,verify=rank==0);dist.barrier(device_ids=[rank])\n        ck,policy=checked_weights(cfg);save(out/f\'runtime_policy_rank{rank}.json\',policy)\n        context=None\n        if cfg.get(\'semantic\'):\n            from csa_patch_model import CSAPatchCaptioner\n            from semantic_data import SemanticContext\n            model=CSAPatchCaptioner(len(words),ck[\'model\'][\'core.nouns\'],ck[\'model\'][\'core.verbs\']);context=SemanticContext(cfg[\'stage\'])\n        else:model=PatchCaptioner(len(words))\n        model=model.to(device);model.load_state_dict(ck[\'model\'],strict=True);del ck;model.eval();rows=[];beams=[];images=np.arange(TEST_SIZE)[rank::2]\n        with torch.inference_mode():\n            for offset in range(0,len(images),8):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'VALID decode phase deadline\')\n                index=images[offset:offset+8];x=torch.from_numpy(np.asarray(data[\'patches\'][index],np.float32)).to(device)\n                memory=model.encode(x,context.images(index,device)) if context else model.encode(x);seq,scores=beam_search(model,memory,width=locked[\'beam_width\'],maximum=locked[\'max_new_tokens\'])\n                if offset==0:check_paths(model,memory,seq,scores)\n                for i, sequences, values in zip(index.tolist(),seq.cpu().tolist(),scores.cpu().tolist()):\n                    beams.append(dict(coco_id=int(data[\'ids\'][i]),candidates=[\n                        dict(ids=tokens,caption=text_from_ids(tokens,words),logprob=float(value))\n                        for tokens,value in zip(sequences,values,strict=True)]))\n                for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                if offset%200==0:print(\'VALID decode rank\',rank,len(rows),\'/\',len(images),flush=True)\n            save(out/f\'captions_rank{rank}.json\',rows)\n            save(out/f\'beams_rank{rank}.json\',beams)\n            nll_sum=0.;tokens=0;ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n            for offset in range(0,len(ev),32):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST teacherNLL phase deadline; captions preserved\')\n                x,seq=caption_batch(data,ev[offset:offset+32]);x=torch.from_numpy(x).to(device);seq=torch.from_numpy(seq).to(device)\n                logits=model(x,seq[:,:-1],context.events(data,ev[offset:offset+32],device)) if context else model(x,seq[:,:-1]);nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n        if len(rows)!=2500:raise ValueError(\'Incomplete TEST rank\')\n        save(out/f\'worker{rank}.json\',dict(complete=True,rank=rank,images=len(rows),NLL_sum=nll_sum,NLL_tokens=tokens,seconds=time.monotonic()-started,training=False,checkpoint_sha256=read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'] if cfg.get(\'evaluate_pilot\') else d.CHECKPOINT_SHA,caption_sha256=sha(out/f\'captions_rank{rank}.json\'),beams_sha256=sha(out/f\'beams_rank{rank}.json\'),runtime_policy=policy,first_GPU_beam_path_check_passed=True))\n        dist.barrier(device_ids=[rank])\n    finally:\n        if dist.is_initialized():dist.destroy_process_group()\n\ndef inputs(cfg):\n    d=contracts(cfg);out=Path(cfg[\'output\']);locked=d.lock();saved=read(out/\'test_lock.json\');d.validate_lock(saved);receipt=read(out/\'stage_receipt.json\')\n    if saved[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\') or sha(out/\'test_references.json\')!=receipt[\'output_files_sha256\'][\'test_references.json\']:raise ValueError(\'Changed TEST references/receipt\')\n    refs=read(out/\'test_references.json\');workers=[read(out/f\'worker{r}.json\') for r in (0,1)]\n    if not all(w[\'complete\'] and w[\'images\']==2500 and w[\'checkpoint_sha256\']==read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'] for w in workers):raise ValueError(\'Incomplete TEST model output\')\n    if workers[0][\'runtime_policy\']!=workers[1][\'runtime_policy\'] or not all(w[\'first_GPU_beam_path_check_passed\'] for w in workers):raise ValueError(\'Runtime/beam guards differ between ranks\')\n    rows=[]\n    for rank,w in enumerate(workers):\n        if sha(out/f\'captions_rank{rank}.json\')!=w[\'caption_sha256\']:raise ValueError(\'Changed TEST rank captions\')\n        rows+=read(out/f\'captions_rank{rank}.json\')\n    check_rows(rows,locked[\'expected_test_ids\'])\n    if len(refs)!=5000 or {r[\'coco_id\'] for r in refs}!=set(locked[\'expected_test_ids\']):raise ValueError(\'Wrong VALID references\')\n    return out,refs,rows,workers\n\ndef tokenize(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    out,refs,rows,workers=inputs(cfg);rows=sorted(rows,key=lambda r:r[\'coco_id\']);save(out/\'captions.json\',rows)\n    tok=PTBTokenizer();gts=read(cfg[\'cached_reference_ptb\']);hyp=tok.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in rows})\n    save(out/\'reference_ptb.json\',gts);save(out/\'caption_ptb.json\',hyp)\n    save(out/\'scoring_lock.json\',dict(checkpoint_sha256=read(out/\'pilot_checkpoint_receipt.json\')[\'sha256\'],files_sha256={n:sha(out/n) for n in (\'captions.json\',\'test_references.json\',\'test_lock.json\',\'reference_ptb.json\',\'caption_ptb.json\')},training=False,reranking=False))\n\ndef score(cfg,mode):\n    out=Path(cfg[\'output\']);sealed=read(out/\'scoring_lock.json\')\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed scoring input \'+n)\n    gts=read(out/\'reference_ptb.json\');hyp=read(out/\'caption_ptb.json\')\n    if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError(\'Incomplete VALID scorer cohort\')\n    result={}\n    if mode==\'basic\':\n        from pycocoevalcap.bleu.bleu import Bleu\n        from pycocoevalcap.cider.cider import Cider\n        from pycocoevalcap.rouge.rouge import Rouge\n        b,_=Bleu(4).compute_score(gts,hyp);c,per=Cider().compute_score(gts,hyp);r,_=Rouge().compute_score(gts,hyp)\n        result=dict(BLEU_1=float(b[0]),BLEU_4=float(b[3]),CIDEr=float(c),ROUGE_L=float(r));save(out/\'cider_per_image.json\',[dict(coco_id=int(i),CIDEr=float(v)) for i,v in zip(gts,per)])\n    elif mode in (\'meteor\',\'spice\'):\n        if mode==\'meteor\':\n            from pycocoevalcap.meteor.meteor import Meteor\n            scorer=Meteor()\n        else:\n            from pycocoevalcap.spice.spice import Spice\n            scorer=Spice()\n        try:point,per=scorer.compute_score(gts,hyp)\n        finally:del scorer\n        if len(per)!=5000:raise ValueError(\'Incomplete \'+mode)\n        result={mode.upper():float(point)}\n        values=[float(x) for x in per] if mode==\'meteor\' else [float(x[\'All\'][\'f\']) for x in per]\n        if not all(math.isfinite(x) for x in values):raise ValueError(\'Nonfinite per-image \'+mode)\n        save(out/(mode+\'_per_image.json\'),[dict(coco_id=int(i),score=v) for i,v in zip(gts,values)])\n    elif mode==\'chair\':\n        rows=read(out/\'captions.json\');save(out/\'chair_input.json\',[dict(image_id=r[\'coco_id\'],caption=r[\'caption\']) for r in rows]);chair=Path(cfg[\'chair\'])\n        subprocess.run([sys.executable,\'-u\',chair/\'chair.py\',\'--cap_file\',out/\'chair_input.json\',\'--cache\',chair/\'chair.pkl\',\'--save_path\',out/\'chair_output.json\'],cwd=chair,check=True,timeout=600)\n        result,counts=chair_counts(read(out/\'chair_output.json\'),rows);save(out/\'chair_counts.json\',counts)\n    else:raise ValueError(mode)\n    if not all(math.isfinite(v) for v in result.values()):raise ValueError(\'Nonfinite metric\')\n    save(out/(mode+\'.json\'),dict(images=5000,metrics=result,input_sha256=sealed[\'files_sha256\']))\n\ndef combine(cfg):\n    out,refs,rows,workers=inputs(cfg);sealed=read(out/\'scoring_lock.json\');metrics={}\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed final scoring input \'+n)\n    for mode in (\'basic\',\'meteor\',\'spice\',\'chair\'):\n        p=read(out/(mode+\'.json\'))\n        if p[\'images\']!=5000 or p[\'input_sha256\']!=sealed[\'files_sha256\']:raise ValueError(\'Incomplete/changed metric phase\')\n        metrics.update(p[\'metrics\'])\n    required={\'BLEU_1\',\'BLEU_4\',\'METEOR\',\'ROUGE_L\',\'CIDEr\',\'SPICE\',\'CHAIRs\',\'CHAIRi\',\'Recall\'}\n    if set(metrics)!=required or not all(math.isfinite(v) for v in metrics.values()):raise ValueError(\'Need all eight metrics and recall\')\n    metrics[\'NLL\']=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers)\n    save(out/\'result.json\',dict(status=\'CSA_PER_CAPTION_BEST9_TEST5000_COMPLETE\' if cfg.get(\'semantic\') else \'CLIP_PATCH_199_TEST5000_COMPLETE\',images=5000,checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,source_version=contracts(cfg).lock()[\'source_version\'],epoch=contracts(cfg).lock()[\'epoch\'],metrics=metrics,display_x100={k:100*v for k,v in metrics.items() if k!=\'NLL\'},beam_width=5,max_new_tokens=20,training=False,reranking=False,test_tuning=False,prior_project_test_exposure=True,chair_counts=read(out/\'chair_counts.json\'),environment=read(out/\'environment.json\'),runtime_policy=workers[0][\'runtime_policy\']))\n    print(\'TEST5000 COMPLETE\',metrics,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'checkpoint_preflight\',\'prepare\',\'worker\',\'environment\',\'smoke\',\'tokenize\',\'basic\',\'meteor\',\'spice\',\'chair\',\'combine\'));p.add_argument(\'--config\',required=True);a=p.parse_args();cfg=read(a.config)\n    if a.mode in (\'environment\',\'smoke\',\'basic\',\'meteor\',\'spice\',\'chair\'):\n        import score_prompt_test_extra as cpu\n        if a.mode==\'environment\':cpu.environment(Path(cfg[\'output\']))\n        else:\n            cpu.java_path(Path(cfg[\'output\']))\n            if a.mode==\'smoke\':cpu.smoke(Path(cfg[\'output\']))\n            else:score(cfg,a.mode)\n    elif a.mode==\'tokenize\':\n        import score_prompt_test_extra as cpu\n        cpu.java_path(Path(cfg[\'output\']));tokenize(cfg)\n    elif a.mode==\'prepare\':contracts(cfg).prepare(cfg)\n    else:globals()[a.mode](cfg)\n', encoding='utf8')
    (OUTPUT / 'source_receipt.json').write_text(json.dumps(dict(commit=REVISION, sha256=HASHES, training_commit=TRAIN_REVISION, training_pins=TRAIN_PINS), indent=2))
    run([sys.executable, '-u', SOURCE / 'test_test199.py'], 120)
    run([sys.executable, '-u', SOURCE / 'test_semantic_test.py'], 120)
    run([sys.executable, '-u', SOURCE / 'cuda_runtime_guard.py', '--output', OUTPUT], 60)
    sys.path.insert(0, str(SOURCE))
    from csa_per_caption_test_data import checkpoint, VERSION, CHECKPOINT_SHA, prompt_inputs
    run([sys.executable, '-m', 'pip', 'install', '-q', '--target', CLI, 'https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'], 600)
    from kaggle_secrets import UserSecretsClient
    token = None
    for secret_name in SECRET_NAMES:
        try:
            value = UserSecretsClient().get_secret(secret_name).strip()
        except Exception:
            continue
        if value:
            token = value
            break
    if not token:
        raise RuntimeError('Enable Kaggle Secret codex or KAGGLE_API_TOKEN with source access.')
    TOKENS.append(token)
    api_env = os.environ.copy()
    for key in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
        api_env.pop(key, None)
    api_env.update(PYTHONPATH=str(CLI), KAGGLE_API_TOKEN=token, PYTHONUNBUFFERED='1')
    run([sys.executable, '-c', 'SOURCE_VERSION=' + repr(SOURCE_VERSION) + '\n' + ROUTING_PREFLIGHT], 120, api_env)
    target = SOURCE / 'download_best'
    target.mkdir()
    run([sys.executable, '-u', '-c', 'from kaggle.cli import main; main()', 'kernels', 'output', SOURCE_VERSION, '-p', str(target), '--file-pattern', '(^|/)best[.]bin$'], 1200, api_env)
    weights = checkpoint(target)
    origin = 'exact Save Version API download'
    cfg['checkpoint_root'] = str(weights)
    write_config()
    (OUTPUT / 'checkpoint_source.json').write_text(json.dumps(dict(version=VERSION, sha256=CHECKPOINT_SHA, path=str(weights), origin=origin), indent=2))
    run([sys.executable, '-u', SOURCE / 'run_csa_per_caption_test.py', 'checkpoint_preflight', '--config', CONFIG], 120)
    XE_CHECKPOINT_SHA = CHECKPOINT_SHA
    training_runtime = json.loads((ROOT_OUTPUT / 'cuda_runtime.json').read_text())
    SCST_TRAIN_TORCH = training_runtime['torch_version']
    (ROOT_OUTPUT / 'training_runtime.json').write_text(json.dumps(training_runtime))
    run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--target', DEPS, 'pycocoevalcap==1.2', 'nltk==3.9.2', 'future'], 600)
    run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--target', DEPS, 'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'], 600)
    os.environ['PYTHONPATH'] = str(DEPS) + os.pathsep + str(SOURCE)
    run([sys.executable, '-c', "import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"], 300)
    (SOURCE / 'scst_cache.py').write_text('"""CPU-build and GPU-consume reusable SCST input cache."""\nfrom collections import Counter, OrderedDict\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport tarfile\n\nfrom real_bank_data import read, save, sha\n\nSCHEMA = \'csa_scst_inputs_v1\'\nSOURCE_SHA = \'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef\'\nVALID_META = (\'test_lock.json\', \'stage_receipt.json\', \'test_references.json\', \'vocabulary.json\')\nASSETS = (\'java8.tar.gz\', \'chair.py\', \'chair.pkl\', \'reference_ptb.json\')\nREWARD = (\'reward_refs.jsonl.gz\', \'reward_df.jsonl.gz\')\n\n\ndef build_reward(stage):\n    from captioning.scst import TrainingCider\n\n    stage = Path(stage)\n    refs = {int(i): v for i, v in read(stage/\'training_references.json\').items()}\n    scorer = TrainingCider(refs, reference_cache_size=2048)\n    for name, values in zip(REWARD, (scorer.references.items(), scorer.df.items())):\n        with gzip.open(stage/name, \'wt\', encoding=\'utf8\') as out:\n            for row in values:\n                out.write(json.dumps(row, ensure_ascii=False)+\'\\n\')\n\n\ndef load_reward(stage):\n    from captioning.scst import TrainingCider\n\n    stage = Path(stage)\n    with gzip.open(stage/REWARD[0], \'rt\', encoding=\'utf8\') as stream:\n        refs = {int(i): tokens for i, tokens in map(json.loads, stream)}\n    with gzip.open(stage/REWARD[1], \'rt\', encoding=\'utf8\') as stream:\n        df = Counter({tuple(key): count for key, count in map(json.loads, stream)})\n    if len(refs) != 113287 or not df:\n        raise ValueError(\'Wrong cached TRAIN reward\')\n    scorer = TrainingCider.__new__(TrainingCider)\n    scorer.references, scorer.df = refs, df\n    scorer.log_count = math.log(len(refs))\n    scorer.reference_cache_size = 2048\n    scorer.vectors = OrderedDict()\n    return scorer\n\n\ndef seal(root, train, valid, meta, images):\n    root, train, valid, meta = map(Path, (root, train, valid, meta))\n    if images not in (4000, 20000, 40000):\n        raise ValueError(\'Unsupported cache schedule size\')\n    (root/\'train\').mkdir()\n    for name in (\'pilot_ids.npy\', \'pilot_sequences.npy\', \'pilot_patches.npy\', *REWARD):\n        shutil.move(train/name, root/\'train\'/name)\n    for path in train.glob(\'sem_*.npy\'):\n        shutil.move(path, root/\'train\'/path.name)\n    (root/\'valid\').mkdir()\n    for path in valid.glob(\'*.npy\'):\n        shutil.move(path, root/\'valid\'/path.name)\n    for name in VALID_META:\n        shutil.copyfile(meta/name, root/name)\n    lock = read(root/\'test_lock.json\')\n    lock.pop(\'checkpoint\', None)  # The consumer binds the actual mounted binary.\n    if lock[\'checkpoint_sha256\'] != SOURCE_SHA:\n        raise ValueError(\'Wrong checkpoint lock\')\n    save(root/\'test_lock.json\', lock)\n    files = {p.relative_to(root).as_posix(): sha(p) for p in sorted(root.rglob(\'*\')) if p.is_file()}\n    save(root/\'scst_cache_receipt.json\', dict(schema=SCHEMA, source_sha256=SOURCE_SHA,\n         train_images=images, train_start=0, train_stop=images, validation_images=5000, test_used=False, files_sha256=files))\n\n\ndef mount(cfg, root, destination, linked_patches=None):\n    root, destination = Path(root), Path(destination)\n    manifest = read(root/\'scst_cache_receipt.json\')\n    if (manifest[\'schema\'] != SCHEMA or manifest[\'source_sha256\'] != SOURCE_SHA\n            or manifest[\'train_images\'] not in (4000, 20000, 40000)\n            or manifest.get(\'train_start\', 0) != 0\n            or manifest[\'validation_images\'] != 5000 or manifest[\'test_used\']):\n        raise ValueError(\'Wrong SCST cache identity\')\n    required = set((*VALID_META, *ASSETS, *(f\'train/{n}\' for n in REWARD),\n                    \'train/pilot_patches.npy\', \'train/pilot_ids.npy\', \'train/pilot_sequences.npy\',\n                    \'train/sem_text_vectors.npy\', \'valid/patches.npy\'))\n    if not required <= manifest[\'files_sha256\'].keys():\n        raise ValueError(\'Incomplete SCST cache\')\n    for name, digest in manifest[\'files_sha256\'].items():\n        path = root/name\n        target = path.resolve()\n        linked = (linked_patches or {}).get(name)\n        if (not target.is_relative_to(root.resolve()) and target != linked) or sha(path) != digest:\n            raise ValueError(\'Changed cache file: \'+name)\n    for name in VALID_META:\n        shutil.copyfile(root/name, destination/name)\n    lock = read(destination/\'test_lock.json\')\n    lock[\'checkpoint\'] = cfg[\'checkpoint_root\']\n    save(destination/\'test_lock.json\', lock)\n    cfg.update(stage=str(root/\'valid\'), train_stage=str(root/\'train\'),\n               cached_reference_ptb=str(root/\'reference_ptb.json\'))\n    save(destination/\'scst_cache_receipt.json\', manifest)\n    chair = Path(cfg[\'chair\'])\n    chair.mkdir()\n    for name in (\'chair.py\', \'chair.pkl\'):\n        shutil.copyfile(root/name, chair/name)\n    jre = destination/\'jre\'\n    with tarfile.open(root/\'java8.tar.gz\') as t:\n        t.extractall(jre, filter=\'data\')\n    java, = jre.rglob(\'bin/java\')\n    save(destination/\'environment.json\', dict(java=str(java), cached=True))\n\n\ndef seal_train_shard(root, train, start, stop):\n    if (start, stop) not in ((40000,80000),(80000,113287)):\n        raise ValueError(\'Wrong full-cache shard range\')\n    root, train = Path(root), Path(train)\n    dest = root/\'train\'\n    dest.mkdir(parents=True)\n    for name in (\'pilot_ids.npy\',\'pilot_sequences.npy\',\'pilot_patches.npy\'):\n        shutil.move(train/name,dest/name)\n    for path in train.glob(\'sem_*.npy\'):\n        shutil.move(path,dest/path.name)\n    files = {p.relative_to(root).as_posix():sha(p) for p in sorted(dest.iterdir())}\n    save(root/\'scst_train_shard_receipt.json\',dict(schema=SCHEMA,source_sha256=SOURCE_SHA,\n         train_images=stop-start,train_start=start,train_stop=stop,validation_images=0,\n         test_used=False,files_sha256=files))\n', encoding='utf8')
    (SOURCE / 'scst_dataset_mount.py').write_text("import hashlib, json, shutil, zipfile\nfrom pathlib import Path\nfrom real_bank_data import read, sha\n\ndef materialize_part0(dataset, local, expected_parts, expected_ids):\n    dataset, local = Path(dataset).resolve(), Path(local)\n    manifest = read(dataset/'cache_manifest.json')\n    if (manifest['schema']!='csa_scst_full_dataset_v1' or manifest['train_images']!=113287\n            or manifest['valid_images']!=5000 or manifest['test_used']\n            or manifest['ordered_train_ids_sha256']!=expected_ids or len(manifest['parts'])!=3):\n        raise ValueError('Wrong full cache Dataset manifest')\n    for part, expected in zip(manifest['parts'],expected_parts,strict=True):\n        if any(part.get(key)!=value for key,value in expected.items()):\n            raise ValueError('Wrong source part identity in Dataset manifest')\n    part = manifest['parts'][0]\n    bundle = dataset/'cache_metadata_zip.bin'\n    if sha(bundle)!=manifest['bundle_sha256']:\n        raise ValueError('Changed cache metadata bundle')\n    patch_names = {'train/pilot_patches.npy':'part0_train_pilot_patches.npy',\n                   'valid/patches.npy':'part0_valid_patches.npy'}\n    links = {}\n    with zipfile.ZipFile(bundle) as archive:\n        name = 'part0/scst_cache_receipt.json'\n        if part['receipt_member']!=name:\n            raise ValueError('Unexpected original receipt path')\n        raw = archive.read(name)\n        if hashlib.sha256(raw).hexdigest()!=expected_parts[0]['receipt_sha256']:\n            raise ValueError('Changed original Part 0 receipt')\n        files=json.loads(raw)['files_sha256']\n        if set(files)!=set(part['files']):\n            raise ValueError('Incomplete Part 0 file map')\n        local.mkdir()\n        (local/'scst_cache_receipt.json').write_bytes(raw)\n        for name,digest in files.items():\n            entry=part['files'][name]\n            target=local/name\n            if not target.resolve().is_relative_to(local.resolve()) or entry['sha256']!=digest:\n                raise ValueError('Wrong mapped file: '+name)\n            target.parent.mkdir(parents=True,exist_ok=True)\n            if name in patch_names:\n                if entry.get('file')!=patch_names[name]:\n                    raise ValueError('Wrong mounted patch file')\n                source=(dataset/patch_names[name]).resolve(strict=True)\n                if not source.is_relative_to(dataset) or not source.is_file():\n                    raise ValueError('Wrong mounted patch path')\n                target.symlink_to(source)\n                links[name]=source\n            else:\n                member='part0/'+name\n                if entry.get('bundle')!=bundle.name or entry.get('member')!=member:\n                    raise ValueError('Wrong bundled member')\n                with archive.open(member) as source, target.open('xb') as dest:\n                    shutil.copyfileobj(source,dest,length=8*1024**2)\n    return links\n", encoding='utf8')
    from scst_cache import mount
    from scst_dataset_mount import materialize_part0
    matches = list(Path('/kaggle/input').rglob('cache_manifest.json')) if not CACHE_ROOT else [Path(CACHE_ROOT) / 'cache_manifest.json']
    matches = [p for p in matches if p.is_file() and (p.parent / 'cache_metadata_zip.bin').is_file()]
    if len(matches) != 1:
        raise ValueError('Attach one merged full SCST Dataset or set CACHE_ROOT')
    began = time.monotonic()
    local = SOURCE / 'dataset_part0'
    links = materialize_part0(matches[0].parent, local, [{'part': 0, 'receipt_name': 'scst_cache_receipt.json', 'receipt_sha256': '3a2caeb5950518ee5d82ac399de0462b5b4d07eaca0e380f7a9f8c49d5395510', 'archive_sha256': '64eb7d3c958fca4996d717c300191022176e5d688ddd4ba6ea94a5f1a30d3946', 'train_start': 0, 'train_stop': 40000, 'ids_sha256': 'f1fd1507bb3dc6b5dd4ddc57a83e723e2610ec7a45e7a6b05813fa633e519480', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/82'}, {'part': 1, 'receipt_name': 'scst_train_shard_receipt.json', 'receipt_sha256': '7ba08f77f38ce48fc8a8a023dbe70ffd6b9bed142f28d54711a9c3ebd0b98a12', 'archive_sha256': 'b9cadfb1f55067b767ed1e32012d5b5bfc7f9f0000c9ff8df0c2ed9676ab461c', 'train_start': 40000, 'train_stop': 80000, 'ids_sha256': '570fab90751d77fda57aebe84a17537a7cba57d3244fc06a5cdcb55ff2c1d885', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/83'}, {'part': 2, 'receipt_name': 'scst_train_shard_receipt.json', 'receipt_sha256': '3b8247a0e3fc24aff0a0b2125ce5fbd372976274c5fcf5e1f46315516824ad5c', 'archive_sha256': 'a78982bb44fbb4ce5668d36f9df47de5da9536834ac379f123349a893fd0c9c8', 'train_start': 80000, 'train_stop': 113287, 'ids_sha256': '22d47e2713dbe61777a97a8104ba4bc2490e253c4ff8fd71d0e871d57b2d38fb', 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'notebook_version': 'ducanh3/notebook7bc83c4c76/84'}], '6165b0fe756398f3625c4504ebe62f25f07657ce0cffc571a2ce7613ada6e263')
    mount(cfg, local, OUTPUT, linked_patches=links)
    from csa_per_caption_test_data import bind_verified_cache
    bind_verified_cache(cfg, 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef')
    from semantic_data import ARRAYS
    train_root = Path(cfg['train_stage'])
    required = [OUTPUT / 'vocabulary.json', Path(cfg['cached_reference_ptb'])]
    required += [train_root / name for name in ('pilot_patches.npy', 'pilot_sequences.npy', 'pilot_ids.npy', 'reward_refs.jsonl.gz', 'reward_df.jsonl.gz', 'sem_text_vectors.npy')]
    required += [train_root / f'sem_{name}.npy' for name in ARRAYS]
    missing = [str(p) for p in required if not p.is_file()]
    if missing:
        raise FileNotFoundError('Missing trainer inputs: ' + ', '.join(missing))
    write_config()
    print('Dataset cache verified in', round(time.monotonic() - began, 1), 's', flush=True)
    cfg['worker_seconds'] = min(1800, PARENT_DEADLINE - time.monotonic() - 1800)
    write_config()
    if cfg['worker_seconds'] < 300:
        raise TimeoutError('Insufficient budget before TEST inference; no automatic extension')
    launch = [sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone', '--nnodes=1', '--nproc_per_node=2', '--max_restarts=0']
    for path, digest in {'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'experiments/csa_paper/scst_pilot_model.py': '0b5e2d384ffc7195c82888f3f7904eac145ba7291ebed27dc8342ed8e1ac9576'}.items():
        target = SOURCE / path if path.startswith('captioning/') else SOURCE / Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/cd3c8eb372f39df4454a2da27735e00e28f7f394/' + path, digest, target)
    pilot = SOURCE / 'run_csa_per_caption_scst_pilot.py'
    pilot.write_text('"""User-run 1000-step SCST-only for CSA patch captioner."""\nimport argparse\nimport copy\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\nfrom full_data import FeatureReader, load_stage, prepare\nfrom real_bank_data import read, save, sha\nfrom semantic_data import SemanticContext, checked_root, record_arrays\n\nUPDATES = 500\nBATCH = 8\n\n\ndef prepare_train(cfg):\n    local = dict(cfg, stage=cfg[\'train_stage\'])\n    prepare(local)  # Metadata only.\n    stage, out = Path(local[\'stage\']), Path(cfg[\'output\'])\n    data, _, _ = load_stage(stage)\n    refs = {int(r[\'cocoid\']): [s[\'raw\'] for s in r[\'sentences\'][:5]]\n            for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\'] in (\'train\', \'restval\')}\n    if len(refs) != 113287 or set(refs) != set(map(int, data[\'ids\'][:113287])):\n        raise ValueError(\'Wrong TRAIN reward cohort\')\n    selected = np.random.default_rng(42).permutation(113287)[:UPDATES*BATCH]\n    ids = data[\'ids\'][selected]\n    first = np.searchsorted(data[\'caption_image\'], selected)\n    sequences = data[\'sequences\'][first]\n    inv = read(Path(__file__).with_name(\'FULL_PROMPT_FRESH_V175_RESULT.json\'))[\'transport_inventory\'][\'cache_files\']\n    files = {n: inv[n] for n in (\'cache_receipt.json\', \'cohort.json\', \'prompt_links.jsonl\',\n                                \'prompt_texts.json\', \'prompt_embeddings.npy\')}\n    root = checked_root(cfg[\'prompt_test_root\'], \'cache_receipt.json\', files)\n    cohort = read(root/\'cohort.json\')\n    if cohort[\'test\'] or set(cohort[\'train\']) != set(refs):\n        raise ValueError(\'Wrong TRAIN prompt cohort\')\n    texts = read(root/\'prompt_texts.json\')\n    vectors = np.load(root/\'prompt_embeddings.npy\', allow_pickle=False)\n    records, wanted = {}, set(map(int, ids))\n    with (root/\'prompt_links.jsonl\').open(encoding=\'utf8\') as stream:\n        for line in stream:\n            row = json.loads(line)\n            cid = row[\'coco_id\']\n            if cid in wanted:\n                if row[\'group\'] != \'train\' or cid in records:\n                    raise ValueError(\'Wrong TRAIN prompt row\')\n                records[cid] = record_arrays(row, texts)\n    if set(records) != wanted:\n        raise ValueError(\'Missing TRAIN prompts\')\n    for key in records[int(ids[0])]:\n        np.save(stage/f\'sem_{key}.npy\', np.stack([records[int(i)][key] for i in ids]))\n    np.save(stage/\'sem_text_vectors.npy\', vectors)\n    reader = FeatureReader(stage)\n    patches = np.lib.format.open_memmap(stage/\'pilot_patches.npy\', mode=\'w+\', dtype=np.float16,\n                                        shape=(len(ids), 196, 768))\n    try:\n        locations = data[\'locations\'][selected]\n        order = np.lexsort((locations[:, 1], locations[:, 0]))\n        for j, idx in enumerate(order):\n            patches[idx] = reader.get(selected[idx])\n            if (j+1) % 500 == 0:\n                print(\'Pilot TRAIN patches\', j+1, \'/\', len(ids), flush=True)\n        patches.flush()\n    finally:\n        reader.close()\n    np.save(stage/\'pilot_ids.npy\', ids)\n    np.save(stage/\'pilot_sequences.npy\', sequences)\n    save(stage/\'training_references.json\', refs)\n    save(out/\'pilot_inputs.json\', dict(updates=UPDATES, image_events=len(ids),\n         reward_images=len(refs), selected_ids=ids.tolist(), test_used=False))\n\n\ndef train(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch import nn\n    from torch.nn import functional as F\n    from datetime import timedelta\n    from captioning.scst import TrainingCider\n    from csa_patch_model import CSAPatchCaptioner\n    from csa_per_caption_test_data import checkpoint, validate_checkpoint_state, configure_valid\n    from scst_pilot_model import action_logits, policy_loss, captions, selected_logps\n    from scst_kl import reference_kl\n\n    rank = int(os.environ[\'LOCAL_RANK\'])\n    if int(os.environ[\'WORLD_SIZE\']) != 2 or torch.cuda.device_count() != 2:\n        raise ValueError(\'Select T4 x2\')\n    torch.set_num_threads(1)\n    torch.cuda.set_device(rank)\n    device = torch.device(\'cuda\', rank)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.manual_seed(4342+rank)\n    dist.init_process_group(\'nccl\', timeout=timedelta(seconds=900))\n    out, stage = Path(cfg[\'output\']), Path(cfg[\'train_stage\'])\n    start = time.monotonic()\n\n    class Policy(nn.Module):\n        def __init__(self, model):\n            super().__init__()\n            self.model = model\n\n        def forward(self, x, p, sample):\n            memory = self.model.encode(x, p)\n            return self.model.decode(memory, sample)\n\n    @torch.no_grad()\n    def rollout(model, memory, sample):\n        seq = torch.ones((len(memory), 1), dtype=torch.long, device=device)\n        active = torch.ones(len(memory), dtype=torch.bool, device=device)\n        masks, selected = [], []\n        for _ in range(20):\n            logits = model.decode(memory, seq)[:, -1]\n            if not torch.isfinite(logits).all():\n                raise ValueError(\'Nonfinite rollout logits\')\n            logp = F.log_softmax(action_logits(logits), -1)\n            token = torch.multinomial(logp.exp(), 1).squeeze(1) if sample else logp.argmax(-1)\n            selected.append(logp.gather(1, token[:, None]).squeeze(1).masked_fill(~active, 0))\n            masks.append(active)\n            seq = torch.cat((seq, token.masked_fill(~active, 0)[:, None]), 1)\n            active = active & token.ne(2)\n            if not active.any():\n                break\n        return seq, torch.stack(masks, 1), torch.stack(selected, 1)\n\n    try:\n        configure_valid(cfg)\n        src = checkpoint(cfg[\'checkpoint_root\'])\n        ck = torch.load(src, map_location=\'cpu\', weights_only=False)\n        validate_checkpoint_state(ck)\n        words = read(out/\'vocabulary.json\')\n        model = CSAPatchCaptioner(len(words), ck[\'model\'][\'core.nouns\'], ck[\'model\'][\'core.verbs\'])\n        model.load_state_dict(ck[\'model\'], strict=True)\n        del ck\n        model.to(device).eval()\n        before_reference = torch.cuda.get_rng_state(device)\n        reference = copy.deepcopy(model).requires_grad_(False).eval()\n        if not torch.equal(before_reference, torch.cuda.get_rng_state(device)):\n            raise ValueError(\'Teacher construction changed sampling RNG\')\n        reference_versions = tuple(p._version for p in reference.parameters())\n        wrapped = nn.parallel.DistributedDataParallel(Policy(model).eval(), device_ids=[rank], broadcast_buffers=False)\n        optimizer = torch.optim.Adam(wrapped.parameters(), lr=5e-6, betas=(.9,.99), eps=1e-8)\n        optimizer_ids = {id(p) for group in optimizer.param_groups for p in group[\'params\']}\n        if any(p.requires_grad or id(p) in optimizer_ids for p in reference.parameters()):\n            raise ValueError(\'Teacher must be frozen and outside optimizer\')\n        torch.cuda.reset_peak_memory_stats(device)\n        from scst_cache import load_reward\n        cider = load_reward(stage)\n        context = SemanticContext(stage)\n        patches = np.load(stage/\'pilot_patches.npy\', mmap_mode=\'r\', allow_pickle=False)\n        ids = np.load(stage/\'pilot_ids.npy\', allow_pickle=False)\n        if len(ids) != 40000 or len(set(map(int, ids[:UPDATES*BATCH]))) != UPDATES*BATCH:\n            raise ValueError(\'Need exact Part 0 training schedule\')\n\n\n        def checkpoint_step(step):\n            if rank == 0:\n                folder = out/f\'update{step:04d}\'\n                folder.mkdir(exist_ok=True)\n                path = folder/\'pilot_checkpoint.bin\'\n                tmp = path.with_suffix(\'.tmp\')\n                torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),\n                                updates=step, source_sha256=sha(src), kl_beta=cfg[\'kl_beta\'], reference_sha256=sha(src)), tmp)\n                os.replace(tmp, path)\n                save(folder/\'pilot_checkpoint_receipt.json\', dict(sha256=sha(path), updates=step,\n                     source_sha256=sha(src), kl_beta=cfg[\'kl_beta\'], reference_sha256=sha(src)))\n            dist.barrier(device_ids=[rank])\n\n        for step in range(UPDATES):\n            stop = torch.tensor(int(time.monotonic()-start > cfg[\'train_seconds\']-120), device=device)\n            dist.all_reduce(stop, op=dist.ReduceOp.MAX)\n            if int(stop):\n                checkpoint_step(step)\n                raise TimeoutError(\'Pilot cap reached; partial checkpoint preserved\')\n            rows = np.arange(step*BATCH+rank*4, step*BATCH+rank*4+4)\n            x = torch.from_numpy(np.asarray(patches[rows], np.float32)).to(device)\n            p = context.images(rows, device)\n            with torch.no_grad():\n                memory = model.encode(x, p)\n                sample, mask, replay = rollout(model, memory, True)\n                greedy, _, _ = rollout(model, memory, False)\n            sampled = cider.score(ids[rows].tolist(), captions(sample, words))\n            baseline = cider.score(ids[rows].tolist(), captions(greedy, words))\n            advantage = torch.from_numpy(sampled-baseline).to(device)\n            optimizer.zero_grad(set_to_none=True)\n            logits = wrapped(x, p, sample[:, :-1])\n            if step == 0:\n                rescored = selected_logps(logits.detach(), sample[:, 1:], mask)\n                torch.testing.assert_close(rescored, replay, rtol=1e-4, atol=3e-4)\n            rl = policy_loss(logits, sample[:, 1:], mask, advantage)\n            if step == 0:\n                before_teacher = torch.cuda.get_rng_state(device)\n            with torch.no_grad():\n                reference_memory = reference.encode(x, p)\n                reference_logits = reference.decode(reference_memory, sample[:, :-1])\n            if step == 0:\n                if not torch.equal(before_teacher, torch.cuda.get_rng_state(device)):\n                    raise ValueError(\'Teacher forward changed sampling RNG\')\n                torch.testing.assert_close(logits.detach(), reference_logits, rtol=1e-4, atol=5e-5)\n            kl = reference_kl(logits, reference_logits, mask)\n            if step == 0 and abs(float(kl.detach())) > 1e-5:\n                raise ValueError(\'Initial teacher/current KL differs\')\n            loss = rl + cfg[\'kl_beta\']*kl\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite SCST loss\')\n            loss.backward()\n            if step == 0:\n                if any(p.grad is not None for p in reference.parameters()):\n                    raise ValueError(\'Teacher received gradients\')\n                save(out/f\'kl_start_guard_rank{rank}.json\',dict(initial_KL=float(kl.detach()),\n                    teacher_frozen=True, teacher_outside_optimizer=True, teacher_rng_unchanged=True,\n                    policy_replay_passed=True, beta=cfg[\'kl_beta\'], torch_version=str(torch.__version__)))\n            norm = nn.utils.clip_grad_norm_(wrapped.parameters(), 1., error_if_nonfinite=True)\n            optimizer.step()\n            if (step+1) % 25 == 0:\n                stats = torch.tensor([float(rl.detach()), float(sampled.mean()),\n                                      float(baseline.mean())], dtype=torch.float64, device=device)\n                dist.all_reduce(stats)\n                kl_stats = torch.tensor([float(kl.detach()),float(loss.detach())],dtype=torch.float64,device=device)\n                dist.all_reduce(kl_stats)\n                if rank == 0:\n                    row = dict(updates=step+1, SCST=float(stats[0])/2,\n                               sample_reward=float(stats[1])/2, greedy_reward=float(stats[2])/2,\n                               grad_norm=float(norm), seconds=time.monotonic()-start)\n                    row.update(KL=float(kl_stats[0])/2, loss=float(kl_stats[1])/2, beta=cfg[\'kl_beta\'],\n                               rank0_peak_memory_bytes=torch.cuda.max_memory_allocated(device))\n                    print(\'SCST+KL pilot\', row, flush=True)\n                    with (out/\'training.jsonl\').open(\'a\', encoding=\'utf8\') as log:\n                        log.write(json.dumps(row, allow_nan=False)+\'\\n\')\n            if (step+1) % 500 == 0:\n                checkpoint_step(step+1)\n        if reference_versions != tuple(p._version for p in reference.parameters()):\n            raise ValueError(\'Teacher parameters mutated\')\n        if any(p.grad is not None for p in reference.parameters()):\n            raise ValueError(\'Teacher has final gradients\')\n        save(out/f\'kl_end_guard_rank{rank}.json\',dict(updates=UPDATES,teacher_versions_unchanged=True,\n            teacher_gradients_absent=True,peak_memory_bytes=torch.cuda.max_memory_allocated(device)))\n    finally:\n        dist.destroy_process_group()\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'mode\', choices=(\'prepare_train\',\'train\'))\n    parser.add_argument(\'config\')\n    args = parser.parse_args()\n    globals()[args.mode](read(args.config))\n', encoding='utf8')
    (SOURCE/'scst_kl.py').write_text('"""User-run KL regularizer and tensor preflight. Do not import locally."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\nfrom torch.nn import functional as F\n\n\ndef reference_kl(logits, reference_logits, mask):\n    """Sequence-sum KL(current||XE), image mean; PAD/BOS excluded, EOS included."""\n    if logits.shape != reference_logits.shape or logits.ndim != 3 or logits.shape[-1] < 3:\n        raise ValueError(\'KL logit shapes differ\')\n    if mask.shape != logits.shape[:2] or mask.dtype != torch.bool:\n        raise ValueError(\'Wrong KL action mask\')\n    current = logits[..., 2:].float()\n    reference = reference_logits.detach()[..., 2:].float()\n    if not torch.isfinite(current).all() or not torch.isfinite(reference).all():\n        raise ValueError(\'Nonfinite supported KL logits\')\n    logp = F.log_softmax(current, dim=-1)\n    logq = F.log_softmax(reference, dim=-1)\n    tokens = (logp.exp() * (logp-logq)).sum(-1)\n    return tokens.masked_fill(~mask, 0).sum(1).mean()\n\n\ndef smoke(output):\n    from scst_pilot_model import policy_loss\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'KL neural preflight needs CUDA\')\n    torch.manual_seed(20261119)\n    device = torch.device(\'cuda\', 0)\n    logits = torch.randn(2, 3, 7, device=device, requires_grad=True)\n    reference = logits.detach().clone().requires_grad_(True)\n    mask = torch.tensor([[True, True, False], [True, False, False]], device=device)\n    actions = torch.tensor([[2, 4, 0], [3, 0, 0]], device=device)\n    advantage = torch.tensor([.3, -.2], device=device)\n    kl = reference_kl(logits, reference, mask)\n    if abs(float(kl)) > 1e-6:\n        raise ValueError(\'Initial equal-policy KL is not zero\')\n    rl = policy_loss(logits, actions, mask, advantage)\n    baseline_grad, = torch.autograd.grad(rl, logits, retain_graph=True)\n    beta0_grad, = torch.autograd.grad(rl+0*kl, logits, retain_graph=True)\n    torch.testing.assert_close(beta0_grad, baseline_grad, rtol=1e-6, atol=1e-7)\n    initial_grad, = torch.autograd.grad(kl, logits)\n    torch.testing.assert_close(initial_grad, torch.zeros_like(initial_grad), rtol=0, atol=1e-6)\n    changed = logits.detach().clone()\n    changed[..., 3] += 1\n    changed.requires_grad_(True)\n    shifted = reference_kl(changed, reference, mask)\n    shifted.backward()\n    if not torch.isfinite(shifted) or float(shifted) <= 0 or not torch.isfinite(changed.grad).all():\n        raise ValueError(\'Shifted KL/gradient invalid\')\n    if reference.grad is not None or not torch.any(changed.grad):\n        raise ValueError(\'KL must update current policy only\')\n    if changed.grad[~mask].abs().max() != 0 or changed.grad[..., :2].abs().max() != 0:\n        raise ValueError(\'Padding/PAD/BOS received KL gradient\')\n    ignored = changed.detach().clone()\n    ignored[..., :2] = float(\'-inf\')\n    torch.testing.assert_close(reference_kl(ignored, reference, mask), shifted.detach())\n    if float(reference_kl(ignored, reference, torch.zeros_like(mask))) != 0:\n        raise ValueError(\'Empty action mask should give zero KL\')\n    report = dict(equal_policy_zero=True, beta0_gradient_parity=True,\n                  teacher_detached=True, shifted_policy_positive=True,\n                  padding_and_special_logits_ignored=True, finite_gradient=True,\n                  torch_version=str(torch.__version__), GPU=torch.cuda.get_device_name(0),\n                  captioner_executed=False)\n    Path(output).write_text(json.dumps(report, indent=2))\n    print(\'KL tensor/CUDA preflight PASS\', report, flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--output\', type=Path, required=True)\n    smoke(parser.parse_args().output)\n',encoding='utf-8')
    (SOURCE/'scst_kl_valid_compare.py').write_text('"""Saved VALID-only pilot comparison; NumPy imported only for user-run bootstrap."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nPOLICY = \'clip_object_fixed\'\nMETRICS = (\'BLEU_1\', \'BLEU_4\', \'METEOR\', \'ROUGE_L\', \'CIDEr\', \'SPICE\', \'CHAIRs\', \'CHAIRi\', \'Recall\', \'NLL\')\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef gates(xe, control, kl):\n    return dict(CIDEr_vs_XE=kl[\'CIDEr\'] >= xe[\'CIDEr\']+.002,\n                CHAIRs_vs_XE=kl[\'CHAIRs\'] <= xe[\'CHAIRs\'],\n                CHAIRi_vs_XE=kl[\'CHAIRi\'] <= xe[\'CHAIRi\'],\n                CIDEr_retained_vs_control=kl[\'CIDEr\'] >= control[\'CIDEr\']-.002,\n                CHAIRs_reduced_vs_control=kl[\'CHAIRs\'] <= control[\'CHAIRs\']-.002,\n                CHAIRi_vs_control=kl[\'CHAIRi\'] <= control[\'CHAIRi\'])\n\n\ndef sample_rows(root, name):\n    folder = Path(root)/name/\'rerank\'\n    per = {r[\'coco_id\']: r[\'CIDEr\'] for r in read(folder/(POLICY+\'_cider_per_image.json\'))}\n    chair = {r[\'image_id\']: r for r in read(folder/(POLICY+\'_chair_output.json\'))[\'sentences\']}\n    if len(per) != 5000 or set(per) != set(chair):\n        raise ValueError(\'Wrong bootstrap VALID IDs\')\n    return {i: (per[i], bool(chair[i][\'mscoco_hallucinated_words\']),\n                len(chair[i][\'mscoco_hallucinated_words\']), len(chair[i][\'mscoco_generated_words\'])) for i in per}\n\n\ndef bootstrap(left, right, replicates=4000):\n    import numpy as np\n    if set(left) != set(right):\n        raise ValueError(\'Bootstrap cohorts differ\')\n    ids = sorted(left)\n    a = np.asarray([left[i] for i in ids], dtype=np.float64)\n    b = np.asarray([right[i] for i in ids], dtype=np.float64)\n    rng = np.random.default_rng(20261119)\n    values = {k: [] for k in (\'CIDEr_x100\', \'CHAIRs_percentage_points\', \'CHAIRi_percentage_points\')}\n    for start in range(0, replicates, 100):\n        take = rng.integers(len(ids), size=(min(100, replicates-start), len(ids)))\n        x, y = a[take], b[take]\n        values[\'CIDEr_x100\'].extend(100*(x[..., 0]-y[..., 0]).mean(1))\n        values[\'CHAIRs_percentage_points\'].extend(100*(x[..., 1]-y[..., 1]).mean(1))\n        values[\'CHAIRi_percentage_points\'].extend(100*(x[..., 2].sum(1)/np.maximum(1,x[..., 3].sum(1))-\n            y[..., 2].sum(1)/np.maximum(1,y[..., 3].sum(1))))\n    return {k: np.quantile(v, [.025, .975]).tolist() for k, v in values.items()}\n\n\ndef run(root):\n    root = Path(root)\n    results = {name: read(root/name/\'result.json\') for name in (\'xe\', \'scst500\', \'kl500\')}\n    if results[\'kl500\'][\'checkpoint\'].get(\'kl_beta\') != 1.0:\n        raise ValueError(\'Pilot beta differs from the locked protocol\')\n    seals = [hashlib.sha256((root/name/\'test_references.json\').read_bytes()).hexdigest() for name in results]\n    if len(set(seals)) != 1 or any(r[\'images\'] != 5000 or r[\'test_used\'] for r in results.values()):\n        raise ValueError(\'Pilot must use identical VALID references, no TEST\')\n    metrics = {name: r[\'metrics\'][POLICY] for name, r in results.items()}\n    screen = gates(metrics[\'xe\'], metrics[\'scst500\'], metrics[\'kl500\'])\n    rows = {name: sample_rows(root, name) for name in results}\n    result = dict(status=\'SCST_KL500_VALID_COMPLETE\', seed=43, split=\'val\', images=5000,\n        test_used=False, training=True, updates=500, beta=1.0,\n        arms=results, display_x100={name: {k: v[k]*(1 if k==\'NLL\' else 100) for k in METRICS} for name,v in metrics.items()},\n        delta_vs_XE={k: metrics[\'kl500\'][k]-metrics[\'xe\'][k] for k in METRICS},\n        delta_vs_SCST500={k: metrics[\'kl500\'][k]-metrics[\'scst500\'][k] for k in METRICS},\n        gates=screen, screen_passed=all(screen.values()),\n        paired_bootstrap_95_ci={name: bootstrap(rows[\'kl500\'], rows[name]) for name in (\'xe\', \'scst500\')},\n        bootstrap_note=\'4000 paired image resamples, fixed corpus CIDEr IDF; conditional on one trained seed and reused VALID, not training-seed uncertainty.\',\n        no_automatic_extension=True, original_XE_replay=read(root/\'xe_replay.json\'))\n    (root/\'result.json\').write_text(json.dumps(result, indent=2, allow_nan=False))\n    print(json.dumps(dict(gates=screen, screen_passed=all(screen.values()), metrics=metrics), indent=2), flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'root\', type=Path)\n    run(parser.parse_args().root)\n',encoding='utf-8')
    run([sys.executable,'-c','SOURCE_VERSION='+repr(CONTROL_VERSION)+'\n'+ROUTING_PREFLIGHT],120,api_env)
    control_download=SOURCE/'control500';control_download.mkdir()
    run([sys.executable,'-u','-c','from kaggle.cli import main; main()','kernels','output',CONTROL_VERSION,
        '-p',control_download,'--file-pattern',r'(^|/)update0500/(pilot_checkpoint[.]bin|pilot_checkpoint_receipt[.]json)$'],1200,api_env)
    control_files=list(control_download.rglob('pilot_checkpoint.bin'))
    if len(control_files)!=1:raise ValueError('Need exact185 update0500 checkpoint')
    control_weights=control_files[0]
    control_receipt=json.loads(control_weights.with_name('pilot_checkpoint_receipt.json').read_text())
    if control_receipt!={'sha256': '8f1a1fc181b335543d8be117da1315260cdc01b4d4fd4b464d8367a82003d500', 'updates': 500, 'source_sha256': 'a51e7ec21be68aa1cda9e0805bd6248add6199914a5c262fce138988066051a5'} or hashlib.file_digest(control_weights.open('rb'),'sha256').hexdigest()!=control_receipt['sha256']:
        raise ValueError('Changed saved SCST500 control')
    (ROOT_OUTPUT/'control_source.json').write_text(json.dumps(dict(source=CONTROL_VERSION,checkpoint=control_receipt)))
    original_predictions=SOURCE/'original_XE';original_predictions.mkdir()
    run([sys.executable,'-u','-c','from kaggle.cli import main; main()','kernels','output',SOURCE_VERSION,
        '-p',original_predictions,'--file-pattern',r'(^|/)evaluations/epoch_08/val_rank[01][.]json$'],900,api_env)
    original_rows=[]
    for name,digest in {'val_rank0.json': 'bfd0d51601c125b2a844910c1562b96ab34e5c2e3ec3099ad725d203b8765e67', 'val_rank1.json': 'f8e5c9e78b1435f85fca99ee679094d0622b6af6377ea5a697283d4beef49960'}.items():
        matches=list(original_predictions.rglob(name))
        if len(matches)!=1 or hashlib.file_digest(matches[0].open('rb'),'sha256').hexdigest()!=digest:
            raise ValueError('Changed original XE VALID caption payload')
        original_rows.extend(json.loads(matches[0].read_text()))
    original_captions={r['coco_id']:r['caption'] for r in original_rows}
    (ROOT_OUTPUT/'original_XE_VALID.json').write_text(json.dumps(original_rows))
    for path,digest in {'captioning/object_consistency.py':'5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73',
        'captioning/multiscore_reranking.py':'d1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443',
        'experiments/csa_paper/RERANK_TEST_INPUTS.json':'33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46'}.items():
        target=SOURCE/path if path.startswith('captioning/') else SOURCE/Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/277851d0fbc4643ae7001bc7702dd0ce5c91d0f6/'+path,digest,target)
    (SOURCE/'csa_per_caption_rerank_data.py').write_text('"""Prepare and check inputs for the frozen CSA-only rerank replay."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom captioning.object_consistency import index_detections\nfrom patch_beam import text_from_ids\nfrom real_bank_data import read, save, sha\nfrom test199_data import DATA_SHA\n\nHERE = Path(__file__).resolve().parent\nPOLICIES = (\'decoder_rank0\', \'clip_object_fixed\')\nDETECTION_SHA = \'498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c\'\n\n\ndef prepare(cfg):\n    out = Path(cfg[\'output\'])\n    test = Path(cfg[\'test_output\'])\n    source = read(test/\'test_lock.json\')\n    policy = read(HERE/\'RERANK_TEST_INPUTS.json\')\n    replay = read(HERE/\'REPLAY_RECEIPTS.json\')\n    receipt = read(test/\'pilot_checkpoint_receipt.json\')\n    if (source[\'checkpoint_sha256\'] != replay[\'xe_source_sha256\']\n            or receipt[\'source_sha256\'] != replay[\'xe_source_sha256\']\n            or receipt[\'sha256\'] != replay[\'checkpoint_sha256\'] or receipt[\'updates\'] != replay[\'updates\']):\n        raise ValueError(\'Changed CSA checkpoint\')\n    if sha(cfg[\'dataset_json\']) != DATA_SHA or sha(cfg[\'detections\']) != DETECTION_SHA:\n        raise ValueError(\'Changed COCO metadata or detector\')\n    records = sorted((r for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\'] == \'val\'),\n                     key=lambda r: int(r[\'cocoid\']))\n    if [int(r[\'cocoid\']) for r in records] != source[\'expected_test_ids\']:\n        raise ValueError(\'Wrong TEST image cohort\')\n    detections = index_detections(read(cfg[\'detections\']))\n    root = Path(cfg[\'dataset_json\']).parent.resolve()\n    names = {r[\'filename\'] for r in records}\n    paths = {}\n    for p in root.rglob(\'*.jpg\'):\n        if p.name in names:\n            if p.name in paths or not p.resolve().is_relative_to(root):\n                raise ValueError(\'Ambiguous TEST JPEG: \'+p.name)\n            paths[p.name] = p\n    if set(paths) != names or not names <= detections.keys():\n        raise ValueError(\'Missing TEST JPEGs or detector rows\')\n    inputs = []\n    for row in records:\n        name = row[\'filename\']\n        path = paths[name]\n        inputs.append(dict(coco_id=int(row[\'cocoid\']), image_path=str(path),\n                           image_sha256=sha(path), detected=detections[name]))\n    save(out/\'inference_inputs.json\', inputs)\n    (out/\'test_references.json\').write_bytes((test/\'test_references.json\').read_bytes())\n    save(out/\'test_lock.json\', dict(\n        checkpoint_sha256=replay[\'checkpoint_sha256\'], images=5000,\n        clip=policy[\'clip\'], selector=policy[\'frozen_selector\'],\n        inference_sha256=sha(out/\'inference_inputs.json\'),\n        references_sha256=sha(out/\'test_references.json\')))\n\n\ndef checked_lock(out):\n    out = Path(out)\n    lock = read(out/\'test_lock.json\')\n    policy = read(HERE/\'RERANK_TEST_INPUTS.json\')\n    replay = read(HERE/\'REPLAY_RECEIPTS.json\')\n    if (lock[\'clip\'], lock[\'selector\'], lock[\'checkpoint_sha256\'], lock[\'images\']) != (\n            policy[\'clip\'], policy[\'frozen_selector\'], replay[\'checkpoint_sha256\'], 5000):\n        raise ValueError(\'Changed rerank policy or cohort\')\n    if sha(out/\'inference_inputs.json\') != lock[\'inference_sha256\']:\n        raise ValueError(\'Changed inference inputs\')\n    return lock\n\n\ndef read_beams(out, lock):\n    out = Path(out)\n    test = out.parent\n    inputs = read(out/\'inference_inputs.json\')\n    words = read(test/\'vocabulary.json\')\n    by_id = {}\n    for rank in (0, 1):\n        path = test/f\'beams_rank{rank}.json\'\n        receipt = read(test/f\'worker{rank}.json\')\n        if (not receipt[\'complete\'] or receipt[\'images\'] != 2500\n                or receipt[\'checkpoint_sha256\'] != lock[\'checkpoint_sha256\']\n                or sha(path) != receipt[\'beams_sha256\']):\n            raise ValueError(\'Incomplete CSA beam replay\')\n        rows = read(path)\n        original = read(test/f\'captions_rank{rank}.json\')\n        if len(original) != 2500 or [r[\'coco_id\'] for r in rows] != [r[\'coco_id\'] for r in inputs[rank::2]]:\n            raise ValueError(\'Wrong TEST beam coverage\')\n        for row, top in zip(rows, original):\n            beams = row[\'candidates\']\n            if (len(beams) != 5 or row[\'coco_id\'] != top[\'coco_id\']\n                    or beams[0] != dict(ids=top[\'ids\'], caption=top[\'caption\'], logprob=top[\'score\'])):\n                raise ValueError(\'Top-1 does not match beam candidates\')\n            if any(c[\'caption\'] != text_from_ids(c[\'ids\'], words) for c in beams):\n                raise ValueError(\'Changed beam text\')\n            by_id[row[\'coco_id\']] = row\n    return [by_id[row[\'coco_id\']] for row in inputs]\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    prepare(read(parser.parse_args().config))\n',encoding='utf-8')
    (SOURCE/'run_csa_per_caption_rerank.py').write_text("import argparse, hashlib, io, json, os, time\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom csa_per_caption_rerank_data import checked_lock, read_beams, POLICIES\nfrom captioning.multiscore_reranking import rerank_multiscore\nfrom score_prompt_test_extra import java_path\n\ndef fixed_rank(row, clipscore, detected):\n    if len(row['candidates']) != 5 or len(clipscore) != 5 or any((not np.isfinite(x) or not 0 <= x <= 2.50001 for x in clipscore)):\n        raise ValueError('Invalid TEST CLIP scores/width')\n    scores = [c['logprob'] for c in row['candidates']]\n    if not all(np.isfinite(scores)) or scores != sorted(scores, reverse=True):\n        raise ValueError('Invalid beam logprobs')\n    enriched = [dict(c, clipscore=float(s)) for c, s in zip(row['candidates'], clipscore)]\n    chosen, _ = rerank_multiscore(dict(candidates=enriched), detected, 0.4, 0.3)\n    return int(chosen['original_rank'])\n\ndef prefetch(cfg):\n    from huggingface_hub import snapshot_download\n    out = Path(cfg['output'])\n    locked = checked_lock(out)['clip']\n    root = Path(snapshot_download(locked['id'], revision=locked['revision'], allow_patterns=list(locked['files_sha256']), cache_dir=cfg['model_cache']))\n    for n, h in locked['files_sha256'].items():\n        if sha(root / n) != h:\n            raise ValueError('CLIP assets differ from VALID164: ' + n)\n    save(out / 'clip_assets.json', dict(**locked, path=str(root)))\n    print('CLIP model/processor match frozen VALID164 bytes; no BLIP download.', flush=True)\n\ndef text_preflight(cfg):\n    from transformers import CLIPProcessor\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    rows = read_beams(out, lock)\n    model = read(out / 'clip_assets.json')\n    processor = CLIPProcessor.from_pretrained(model['path'], local_files_only=True, use_fast=False)\n    lengths = [len(v) for v in processor.tokenizer([c['caption'] for r in rows for c in r['candidates']], truncation=False)['input_ids']]\n    if len(lengths) != 25000 or max(lengths) > 77:\n        raise ValueError('Would truncate TEST candidates')\n    save(out / 'token_coverage.json', dict(images=5000, captions=25000, max_tokens=max(lengths), limit=77))\n\ndef worker(cfg):\n    import torch\n    import torch.nn.functional as F\n    from PIL import Image\n    from transformers import CLIPModel, CLIPProcessor\n    rank = int(os.environ['LOCAL_RANK'])\n    if int(os.environ['WORLD_SIZE']) != 2 or torch.cuda.device_count() != 2:\n        raise ValueError('Select T4x2')\n    torch.cuda.set_device(rank)\n    device = torch.device('cuda', rank)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    asset = read(out / 'clip_assets.json')\n    for n, h in lock['clip']['files_sha256'].items():\n        if sha(Path(asset['path']) / n) != h:\n            raise ValueError('Changed CLIP model/processor')\n    base = CLIPModel.from_pretrained(asset['path'], local_files_only=True, use_safetensors=False, attn_implementation='eager').float().to(device).eval().requires_grad_(False)\n    processor = CLIPProcessor.from_pretrained(asset['path'], local_files_only=True, use_fast=False)\n    if sha(out / 'inference_inputs.json') != lock['inference_sha256']:\n        raise ValueError('Changed TEST inference inputs')\n    inputs = read(out / 'inference_inputs.json')\n    beams = read_beams(out, lock)\n    started = time.monotonic()\n    count = 0\n    path = out / f'scores_rank{rank}.jsonl'\n    with path.open('x', encoding='utf-8') as log, torch.inference_mode():\n        for row, beam in list(zip(inputs, beams))[rank::2]:\n            if time.monotonic() - started > cfg['phase_seconds']:\n                raise TimeoutError('TEST CLIP scoring cap; preserve partial scores')\n            raw = Path(row['image_path']).read_bytes()\n            if hashlib.sha256(raw).hexdigest() != row['image_sha256']:\n                raise ValueError('Changed raw TEST image')\n            with Image.open(io.BytesIO(raw)) as image:\n                image = image.convert('RGB')\n            pixels = processor(images=image, return_tensors='pt')['pixel_values'].to(device)\n            text = processor.tokenizer([c['caption'] for c in beam['candidates']], padding=True, truncation=False, return_tensors='pt').to(device)\n            image_features = F.normalize(base.get_image_features(pixel_values=pixels).float(), dim=-1)\n            text_features = F.normalize(base.get_text_features(**text).float(), dim=-1)\n            clipscore = (2.5 * (image_features * text_features).sum(-1).clamp_min(0)).cpu().tolist()\n            chosen = fixed_rank(beam, clipscore, row['detected'])\n            log.write(json.dumps(dict(coco_id=row['coco_id'], image_sha256=row['image_sha256'], clipscore=clipscore, selected_rank=chosen), allow_nan=False) + '\\n')\n            log.flush()\n            count += 1\n            if count % 100 == 0:\n                print('TEST CLIP GPU', rank, count, '/2500', flush=True)\n            if count == 25 and (time.monotonic() - started) / count * (2500 - count) > cfg['phase_seconds'] - (time.monotonic() - started):\n                raise TimeoutError('Measured TEST CLIP projection exceeds cap')\n    save(out / f'worker{rank}.json', dict(complete=True, images=count, scored_pairs=count * 5, scores_sha256=sha(path), seconds=time.monotonic() - started, torch_version=torch.__version__, training=False, clip_revision=lock['clip']['revision']))\n\ndef score(cfg):\n    from pycocoevalcap.meteor.meteor import Meteor\n    from pycocoevalcap.spice.spice import Spice\n    java_path(Path(cfg['test_output']))\n\n    import subprocess, sys\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    from pycocoevalcap.cider.cider import Cider\n    from pycocoevalcap.bleu.bleu import Bleu\n    from pycocoevalcap.rouge.rouge import Rouge\n    from saved_quality_data import chair_counts\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    inputs = read(out / 'inference_inputs.json')\n    beams = read_beams(out, lock)\n    if sha(out / 'inference_inputs.json') != lock['inference_sha256'] or sha(out / 'test_references.json') != lock['references_sha256']:\n        raise ValueError('Changed locked inputs')\n    scored = {}\n    for rank in (0, 1):\n        w = read(out / f'worker{rank}.json')\n        path = out / f'scores_rank{rank}.jsonl'\n        if not w['complete'] or w['images'] != 2500 or w['scored_pairs'] != 12500 or (sha(path) != w['scores_sha256']):\n            raise ValueError('Incomplete TEST scoring worker')\n        rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]\n        if [r['coco_id'] for r in rows] != [r['coco_id'] for r in inputs[rank::2]]:\n            raise ValueError('Wrong TEST score IDs')\n        scored.update({r['coco_id']: r for r in rows})\n    if len(scored) != 5000:\n        raise ValueError('Missing TEST scores')\n    chosen = []\n    for row, beam in zip(inputs, beams):\n        s = scored[row['coco_id']]\n        if s['image_sha256'] != row['image_sha256']:\n            raise ValueError('Wrong image-score identity')\n        k = fixed_rank(beam, s['clipscore'], row['detected'])\n        if k != s['selected_rank']:\n            raise ValueError('Fixed selection replay differs')\n        chosen.append(k)\n    save(out / 'selected_ranks.json', chosen)\n    refs = read(out / 'test_references.json')\n    ptb = PTBTokenizer()\n    ref = ptb.tokenize({r['coco_id']: [dict(caption=c) for c in r['references']] for r in refs})\n    chair = Path(cfg['chair'])\n    metrics = {}\n    flags = {}\n    if sha(chair / 'chair.pkl') != '73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':\n        raise ValueError('Changed CHAIR cache')\n    for policy in POLICIES:\n        selected = [dict(coco_id=r['coco_id'], caption=r['candidates'][0 if policy == 'decoder_rank0' else k]['caption']) for r, k in zip(beams, chosen)]\n        hyp = ptb.tokenize({r['coco_id']: [dict(caption=r['caption'])] for r in selected})\n        c, per = Cider().compute_score(ref, hyp)\n        b, _ = Bleu(4).compute_score(ref, hyp)\n        r, _ = Rouge().compute_score(ref, hyp)\n        ci = out / (policy + '_chair_input.json')\n        co = out / (policy + '_chair_output.json')\n        save(ci, [dict(image_id=r['coco_id'], caption=r['caption']) for r in selected])\n        subprocess.run([sys.executable, '-u', chair / 'chair.py', '--cap_file', ci, '--cache', chair / 'chair.pkl', '--save_path', co], cwd=chair, check=True, timeout=300)\n        ch = read(co)\n        m, counts = chair_counts(ch, selected)\n        meteor = Meteor()\n        try:\n            meteor_score, meteor_per = meteor.compute_score(ref, hyp)\n        finally:\n            del meteor\n        spice_score, spice_per = Spice().compute_score(ref, hyp)\n        if len(meteor_per) != 5000 or len(spice_per) != 5000 or not np.isfinite([meteor_score, spice_score]).all():\n            raise ValueError('Incomplete METEOR/SPICE')\n        metrics[policy] = dict(CIDEr=float(c), BLEU_1=float(b[0]), BLEU_4=float(b[3]), METEOR=float(meteor_score), SPICE=float(spice_score), ROUGE_L=float(r), **m, counts=counts, mean_words=sum((len(r['caption'].split()) for r in selected)) / 5000, short_le2=sum((len(r['caption'].split()) <= 2 for r in selected)))\n        save(out / (policy + '_captions.json'), selected)\n        save(out / (policy + '_cider_per_image.json'), [dict(coco_id=int(i), CIDEr=float(v)) for i, v in zip(ref, per)])\n        flags[policy] = {r['image_id']: bool(r['mscoco_hallucinated_words']) for r in ch['sentences']}\n    a, b = [flags[p] for p in POLICIES]\n    changes = dict(changed_captions=sum((k != 0 for k in chosen)), rank_histogram={str(k): chosen.count(k) for k in range(5)}, CHAIR_repairs=sum((a[i] and (not b[i]) for i in a)), CHAIR_new_flags=sum((not a[i] and b[i] for i in a)))\n    result = dict(status='CSA_PER_CAPTION_FIXED_RERANK_VALID5000_COMPLETE', images=5000, test_used=False, training=False, checkpoint_sha256=lock['checkpoint_sha256'], metrics=metrics, changes=changes, selector=lock['selector'], tuned_on_test=False, oracle_used_for_selection=False, prior_project_test_exposure=True, selection='both prelocked policies reported; fixed VALID164 policy', NLL_note='same frozen captioner; reranking changes output only, no target-conditioned pass')\n    save(out / 'result.json', result)\n    print(json.dumps(result, indent=2), flush=True)\n\nif __name__ == '__main__':\n    parser = argparse.ArgumentParser()\n    parser.add_argument('mode', choices=('prefetch', 'text_preflight', 'worker', 'score'))\n    parser.add_argument('config')\n    args = parser.parse_args()\n    globals()[args.mode](read(args.config))\n",encoding='utf-8')
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,
        'transformers==4.57.1','tokenizers==0.22.1','huggingface-hub==0.36.2','safetensors==0.6.2'],300)
    evaluator=SOURCE/'run_csa_per_caption_test.py'
    for mode,cap in (('environment',1200),('smoke',600)):
        run([sys.executable,'-u',evaluator,mode,'--config',CONFIG],cap)
    run([sys.executable,'-u',SOURCE/'scst_kl.py','--output',ROOT_OUTPUT/'kl_tensor_preflight.json'],180)
    bootstrap_smoke="from scst_kl_valid_compare import bootstrap; r=bootstrap({0:(.2,1,1,2),1:(.2,1,1,2)},{0:(.1,0,0,2),1:(.1,0,0,2)},20); assert all(abs(x-v)<1e-8 for k,v in [('CIDEr_x100',10),('CHAIRs_percentage_points',100),('CHAIRi_percentage_points',50)] for x in r[k]); print('Paired bootstrap arithmetic PASS')"
    run([sys.executable,'-c',bootstrap_smoke],120)
    base_cfg=dict(cfg)
    xe_weights=weights
    xe_receipt=dict(sha256=XE_CHECKPOINT_SHA,source_sha256=XE_CHECKPOINT_SHA,updates=0)

    def evaluate_arm(name, path, receipt, training_torch):
        folder=ROOT_OUTPUT/name;folder.mkdir()
        for file in ('vocabulary.json','stage_receipt.json','test_lock.json','test_references.json','environment.json'):
            shutil.copyfile(ROOT_OUTPUT/file,folder/file)
        (folder/'training_runtime.json').write_text(json.dumps(dict(torch_version=training_torch)))
        (folder/'pilot_checkpoint_receipt.json').write_text(json.dumps(receipt))
        evaluation=dict(base_cfg,output=str(folder),checkpoint_root=str(xe_weights),
            evaluate_pilot=receipt['updates']!=0,eval_update=receipt['updates'],eval_checkpoint=str(path))
        evaluation['worker_seconds']=min(1800,PARENT_DEADLINE-time.monotonic()-900)
        if evaluation['worker_seconds']<300:raise TimeoutError('Insufficient VALID inference budget')
        config=SOURCE/(name+'_config.json');config.write_text(json.dumps(evaluation))
        (folder/'protocol.json').write_text(json.dumps(evaluation))
        run([sys.executable,'-u',evaluator,'checkpoint_preflight','--config',config],120)
        run([*launch,evaluator,'worker','--config',config],evaluation['worker_seconds']+30)
        (SOURCE/'REPLAY_RECEIPTS.json').write_text(json.dumps(dict(checkpoint_sha256=receipt['sha256'],
            xe_source_sha256=XE_CHECKPOINT_SHA,updates=receipt['updates'])))
        rerank=folder/'rerank';rerank.mkdir()
        rcfg=dict(output=str(rerank),test_output=str(folder),dataset_json=str(COCO_JSON),
            detections=str(DETECTIONS),chair=str(CHAIR),model_cache=str(SOURCE/'hf_cache'),phase_seconds=2100)
        rconfig=SOURCE/(name+'_rerank_config.json');rconfig.write_text(json.dumps(rcfg))
        run([sys.executable,'-u',SOURCE/'csa_per_caption_rerank_data.py',rconfig],900)
        script=SOURCE/'run_csa_per_caption_rerank.py'
        for mode,cap in (('prefetch',1200),('text_preflight',180)):
            run([sys.executable,'-u',script,mode,rconfig],cap)
        run([*launch,script,'worker',rconfig],2130)
        run([sys.executable,'-u',script,'score',rconfig],3600)
        result=json.loads((rerank/'result.json').read_text())
        if result['status']!='CSA_PER_CAPTION_FIXED_RERANK_VALID5000_COMPLETE' or result['checkpoint_sha256']!=receipt['sha256']:
            raise ValueError('Wrong arm VALID checkpoint/result')
        workers=[json.loads((folder/f'worker{rank}.json').read_text()) for rank in (0,1)]
        nll=sum(w['NLL_sum'] for w in workers)/sum(w['NLL_tokens'] for w in workers)
        for metric in result['metrics'].values():metric['NLL']=nll
        result.update(arm=name,checkpoint=receipt,NLL_note='Same frozen captioner teacher-forced NLL for both selections.')
        (folder/'result.json').write_text(json.dumps(result,indent=2,allow_nan=False))
        return result

    evaluate_arm('xe',xe_weights,xe_receipt,'2.11.0+cu128')
    current_rows=json.loads((ROOT_OUTPUT/'xe/rerank/decoder_rank0_captions.json').read_text())
    if len(original_captions)!=5000 or set(original_captions)!={r['coco_id'] for r in current_rows}:
        raise ValueError('Original/current XE VALID cohorts differ')
    changed=sum(r['caption']!=original_captions[r['coco_id']] for r in current_rows)
    (ROOT_OUTPUT/'xe_replay.json').write_text(json.dumps(dict(images=5000,changed_captions=changed,
        byte_identical_decode_claimed=False,reference='same cached evaluator across all pilot arms')))
    evaluate_arm('scst500',control_weights,control_receipt,'2.11.0+cu128')
    training=ROOT_OUTPUT/'train';training.mkdir()
    shutil.copyfile(ROOT_OUTPUT/'vocabulary.json',training/'vocabulary.json')
    cfg=dict(base_cfg,output=str(training),checkpoint_root=str(xe_weights),kl_beta=KL_BETA)
    cfg['train_seconds']=min(1800,PARENT_DEADLINE-time.monotonic()-1800)
    if cfg['train_seconds']<300:raise TimeoutError('Insufficient KL500 budget; baseline outputs preserved')
    write_config()
    run([*launch,pilot,'train',CONFIG],cfg['train_seconds']+30)
    final=training/'update0500'
    receipt=json.loads((final/'pilot_checkpoint_receipt.json').read_text())
    if receipt['updates']!=500 or receipt['source_sha256']!=XE_CHECKPOINT_SHA or receipt['kl_beta']!=KL_BETA or receipt['reference_sha256']!=XE_CHECKPOINT_SHA:
        raise ValueError('Wrong KL endpoint/reference')
    for rank in (0,1):
        guard=json.loads((training/f'kl_end_guard_rank{rank}.json').read_text())
        if not guard['teacher_versions_unchanged'] or not guard['teacher_gradients_absent']:raise ValueError('Teacher mutation')
    evaluate_arm('kl500',final/'pilot_checkpoint.bin',receipt,SCST_TRAIN_TORCH)
    run([sys.executable,'-u',SOURCE/'scst_kl_valid_compare.py',ROOT_OUTPUT],600)
    complete=json.loads((ROOT_OUTPUT/'result.json').read_text())['status']=='SCST_KL500_VALID_COMPLETE'
except BaseException:
    (ROOT_OUTPUT/'failure.txt').write_text(redact(traceback.format_exc()));raise
finally:
    (ROOT_OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=True,test_used=False,
        seed=43,updates=500,beta=KL_BETA,xe_source=SOURCE_VERSION,control_source=CONTROL_VERSION)))
    review=ROOT_OUTPUT.with_name(ROOT_OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for path in ROOT_OUTPUT.rglob('*'):
            if path.is_file() and path.suffix in ('.json','.jsonl','.log','.txt'):z.write(path,str(path.relative_to(ROOT_OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP ENTIRE OUTPUT:',ROOT_OUTPUT,flush=True)
    print('VALID pilot complete; no TEST or automatic extension.' if complete else 'INCOMPLETE; preserve artifacts. No automatic restart.',flush=True)
